# 프롬프트 엔지니어링


**학습 목표**

> 1. 프롬프트 엔지니어링의 핵심이 단순한 문장 꾸미기가 아니라 **작업 지시를 명확하게 설계하는 것**임을 이해한다.
> 2. 역할, 목표, 맥락, 제약 조건, 출력 형식, 예시를 조합해 프롬프트를 구성한다.
> 3. zero-shot, few-shot, 출력 형식 지정, 근거 요청, 자체 점검 같은 대표 기법을 실습한다.
> 4. `ChatPromptTemplate`, `StrOutputParser`, `with_structured_output`, `RunnableParallel` 등을 활용해 프롬프트 워크플로우를 만든다.
> 5. **좋은 프롬프트를 작성하고, 테스트하고, 개선하는 반복 절차**를 익힌다.



# 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [1]:
# 필요한 라이브러리 설치
%pip install -U langchain langchain-core langchain-openai python-dotenv pydantic


   ---------------------------------------- 0.0/554.3 kB ? eta -:--:--
   ---------------------------------------- 554.3/554.3 kB 7.0 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.5
    Uninstalling langchain-core-1.4.5:
      Successfully uninstalled langchain-core-1.4.5
Note: you may need to restart the kernel to use updated packages.


## (2) 라이브러리 Import


In [2]:
import os
from pathlib import Path
from getpass import getpass
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)


## (3) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
OPENAI_API_KEY=sk-...

```


In [3]:
load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## (4) 모델 준비
- 프롬프트 엔지니어링 실습에서는 모델 자체보다 **입력 지시를 어떻게 구성하는지**에 집중


In [4]:
MODEL = 'gpt-4.1-mini'

model = ChatOpenAI(
    model=MODEL,
    timeout=60,
    max_retries=3,
)

parser = StrOutputParser()

model


ChatOpenAI(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.7'}}, output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001B65078EA50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001B65078F4D0>, root_client=<openai.OpenAI object at 0x000001B64FC8DFD0>, root_async_client=<openai.AsyncOpenA

# 2. 프롬프트 엔지니어링이란
- 프롬프트 엔지니어링은 LLM에게 주는 지시를 설계해 결과의 품질, 일관성, 재사용성을 높이는 방법
- 나쁜 프롬프트는 보통 다음 문제가 있음

    - 작업 목표가 모호하다.
    - 대상 독자나 상황이 없다.
    - 출력 형식이 정해져 있지 않다.
    - 금지 사항이나 제약 조건이 없다.
    - 좋은 답변의 기준이 없다.

In [ ]:
vague_prompt = ChatPromptTemplate.from_messages([
    ("user", ""),
])

vague_chain = vague_prompt | model | parser

print(vague_chain.invoke({}))


In [ ]:
specific_prompt = ChatPromptTemplate.from_messages([
    ("system", ""),
    (
        "user",
        ,
    ),
])

specific_chain = specific_prompt | model | parser

print(specific_chain.invoke({}))


## [실습] 모호한 요청 개선하기

아래의 모호한 요청을 더 구체적인 프롬프트로 바꿔 실행해 봅니다.

요청 예시:

```text
보고서 써줘
```

개선할 때는 대상 독자, 목적, 길이, 형식, 포함할 내용을 명시합니다.


# 3. 프롬프트의 기본 구성 요소

- 좋은 프롬프트는 보통 다음 요소를 포함

| 구성 요소 | 설명 | 예시 |
|---|---|---|
| 역할 | 모델이 맡을 관점 | `너는 고객 상담 QA 담당자다` |
| 목표 | 수행할 작업 | `문의 내용을 분류한다` |
| 맥락 | 필요한 배경 정보 | `우리 서비스는 B2B SaaS다` |
| 제약 | 하지 말아야 할 것, 길이, 톤 | `추측하지 말고 모르면 모른다고 답한다` |
| 출력 형식 | 결과 구조 | `JSON으로 출력한다` |
| 예시 | 원하는 답변 패턴 | 입력/출력 예시 2~3개 |


In [28]:
component_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "너는 B2B SaaS 고객 문의를 정리하는 고객 성공 매니저다. "
        "답변은 차분하고 업무용 문체로 작성한다.",
    ),
    (
        "user",
        "문의 내용:\n{customer_message}\n\n"
        "작업:\n"
        "1. 문의 유형을 하나로 분류한다: 결제, 버그, 기능 요청, 사용법, 기타\n"
        "2. 고객에게 보낼 3문장 이내의 답변 초안을 작성한다.\n"
        "3. 내부 담당자가 확인해야 할 후속 조치를 2개 작성한다.\n\n"
        "출력 형식:\n"
        "[문의 유형]\n...\n\n[고객 답변]\n...\n\n[후속 조치]\n- ...\n- ...",
    ),
])

component_chain = component_prompt | model | parser

print(component_chain.invoke({
    "customer_message": "갑자기 제 삼성 MASTER 신용 카드 해외 결제가 500만원 됐어요. 환불 무조건 해주시고 경위 파악 부탁드립니다."
}))


[문의 유형]  
결제

[고객 답변]  
안녕하세요, 갑작스러운 해외 결제 건으로 불편을 드려 죄송합니다. 신속히 환불 처리 여부와 결제 내역을 확인하여 안내드리겠습니다. 추가 확인을 위해 조금만 기다려 주시길 부탁드립니다.

[후속 조치]  
- 결제 부서에 해당 거래 내역 및 결제 경위 확인 요청  
- 환불 처리 가능 여부 및 진행 절차 확인 후 고객에게 결과 안내


In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

ex_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content='너는 B2B 고객 관리 매니저이다.'),
    HumanMessage(content='문의 내용:\n {customer_message}')
])

# 4. 프롬프트 템플릿화
- 프롬프트 템플릿은 반복되는 작업에서 바뀌는 부분만 변수로 분리하는 방법


    - 같은 작업을 여러 입력에 반복 적용할 수 있다.
    - 프롬프트 구조를 재사용할 수 있다.
    - 변수명만 보면 어떤 정보가 필요한지 알 수 있다.


In [7]:
report_prompt = PromptTemplate.from_template(
'''
다음 정보를 바탕으로 업무 보고서 초안을 작성하세요.

주제: {topic}
대상 독자: {audience}
보고 목적: {purpose}
핵심 데이터: {data_points}
분량: {length}

작성 조건:
- 제목을 포함하세요.
- 핵심 요약을 먼저 작성하세요.
- 데이터에서 확인되지 않는 내용은 추측하지 마세요.
- 마지막에  추가 확인이 필요한 항목을 작성하세요.
'''
)

formatted_prompt = report_prompt.format(
    topic='신규 고객 온보딩 개선안',
    audience='팀장',
    purpose='다음 분기 개선 과제 제안',
    data_points='첫 주 이탈률 18%',
    length='A4 반 페이지'
)

print(formatted_prompt)



다음 정보를 바탕으로 업무 보고서 초안을 작성하세요.

주제: 신규 고객 온보딩 개선안
대상 독자: 팀장
보고 목적: 다음 분기 개선 과제 제안
핵심 데이터: 첫 주 이탈률 18%
분량: A4 반 페이지

작성 조건:
- 제목을 포함하세요.
- 핵심 요약을 먼저 작성하세요.
- 데이터에서 확인되지 않는 내용은 추측하지 마세요.
- 마지막에  추가 확인이 필요한 항목을 작성하세요.



In [12]:
report_chain = report_prompt | model | parser

print(report_chain.invoke({
    'topic':'신규 고객 온보딩 개선안',
    'audience':'팀장',
    'purpose':'다음 분기 개선 과제 제안',
    'data_points':'첫 주 이탈률 18%',
    'length':'A4 반 페이지'
}))


신규 고객 온보딩 개선안 보고서

1. 핵심 요약  
현재 신규 고객 온보딩 과정에서 첫 주 이탈률이 18%로 나타났습니다. 이 수치는 초기 고객 경험에 개선이 필요함을 시사하며, 다음 분기 내 온보딩 프로세스의 효율성 향상을 위한 구체적인 개선 과제 도출이 필요합니다.

2. 현황 및 문제점  
- 첫 주 이탈률: 18%  
- 상세 원인 분석 데이터 미보유

3. 제안 사항  
- 현재 수집 가능한 데이터 내 이탈 원인을 정확히 파악하기 위한 추가 분석 필요  
- 온보딩 단계별 고객 반응 및 이탈 시점 데이터 수집 및 모니터링 강화  
- 고객 교육 및 지원 콘텐츠 점검 및 보완 예정

4. 추가 확인이 필요한 항목  
- 고객 이탈의 구체적인 이유 및 상황  
- 온보딩 프로세스 각 단계별 상세 이탈률  
- 고객 피드백 및 만족도 조사 결과  
- 타사 온보딩 프로세스 및 벤치마킹 자료

이상입니다.


## [실습] 내 업무용 템플릿 만들기

아래 빈칸을 채워 반복해서 쓸 수 있는 프롬프트 템플릿을 만들어 봅니다.


In [29]:
my_template = PromptTemplate.from_template(
    """
작업 주제: {topic}
대상: {audience}
원하는 결과물: {output_type}
반드시 포함할 내용: {must_include}
제외할 내용: {exclude}
출력 형식: {format}
""".strip()
)

my_chain = my_template | model | parser
print(my_chain.invoke({
    "topic": "스페이스X 여론 분석",
    "audience": "토스증권 이용자",
    "output_type": "스페이스X에 대한 여론을 모든 웹사이트 및 블로그를 조사해서 구매할 것인지 구매하지 않을 것인지 감성 분석",
    "must_include": "어떤 종목을 매도하고 스페이스X를 구매하려고 하는지",
    "exclude": "주가 예측",
    "format": "제목, 상세 내용, 결론",
}))


### 스페이스X 여론 분석 및 투자 전략 제안 – 토스증권 이용자 대상

---

#### 상세 내용

최근 스페이스X에 대한 관심이 급증하면서 각종 웹사이트, 뉴스 포털, 블로그, 커뮤니티에서 다양한 의견들이 활발히 공유되고 있습니다. 토스증권 이용자들의 투자 판단에 도움될 수 있도록, 주요 온라인상에서의 스페이스X 관련 여론과 감성 분석 결과를 정리했습니다.

1. **긍정적 여론**
   - **혁신성과 성장 가능성 부각**: 다수의 블로그와 투자 커뮤니티에서는 스페이스X가 우주 산업의 선도 주자로서 민간 우주여행, 위성 인터넷(스타링크) 등 혁신적인 기술을 개발 중이라는 점을 칭찬하고 있습니다.
   - **장기적 비전과 정부 계약**: 정부와의 지속적인 협력 계약 및 대규모 위성 발사 프로젝트는 안정적인 수익원으로 작용할 전망이라는 긍정적인 평가가 많습니다.
  
2. **부정적 여론**
   - **높은 리스크와 경쟁 심화 우려**: 기술적 실패 가능성, 높은 투자 비용, 우주 항공 분야 내 경쟁 심화에 대한 우려가 존재합니다.
   - **시장 불확실성**: 일부 의견은 아직 수익 구조가 명확하지 않아 초기 투자자에게는 불안 요인이라고 지적합니다.

3. **감성 분석 결과 요약**
   - 전체 게시물 대비 약 70%는 긍정적 감정을 보였으며, 20%는 중립, 10%는 부정적 감정을 표현했습니다.
   - 긍정적 의견에서 ‘혁신’, ‘성장’, ‘미래 산업’ 등 긍정적 키워드가 출현했고, 부정적 의견에서는 ‘불확실’, ‘위험’, ‘과대 평가’ 등이 언급되었습니다.

4. **기존 보유 종목 매도 제안**
   - 현재 토스증권 이용자들이 인기가 높은 일부 전통 항공사 및 방산업체(예: 대한항공, 한화에어로스페이스) 주식은 우주산업으로의 전환 지연과 경쟁 심화 우려로 인해 단기 매도 관점에서 재검토가 권고됩니다.
   - 해당 종목들은 스페이스X가 주도하는 민간 우주 시장에서의 상대적 경쟁력 저하 가능성이 커 투자 소재로서 재평가될 필요가 있습니다

# 5. 출력 형식 지정하기
- LLM 앱에서는 답변을 단순 문장으로 받는 것보다, 프로그램에서 바로 사용할 수 있는 구조로 받는 것이 중요함
- 출력 형식을 지정하는 방법은 크게 두 가지

    1. 프롬프트에 JSON 형식을 직접 명시한다.
    2. Pydantic 모델과 `with_structured_output()`을 사용해 구조화 출력을 요청한다.


In [ ]:
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 문장을 분석해 JSON으로 출력하는 분석기다.'),
    (
        "user",
        ,
    ),
])

json_chain = json_prompt | model | parser

print(json_chain.invoke({"text": "응답 속도는 빠르지만 설명이 조금 부족했어요."}))


In [11]:
# 2. Pydantic 모델과 with_structured_output()을 사용해 구조화 출력 요청
class SentimentResult(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감성 분류 결과")
    reason: str = Field(description="분류 이유")
    confidence: float = Field(description="0.0부터 1.0 사이의 확신도")


sentiment_model = model.with_structured_output(SentimentResult, method="json_schema")

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 고객 문장을 감성 분석하는 분석가다."),
    ("user", "문장: {text}"),
])

sentiment_chain = sentiment_prompt | sentiment_model

sentiment_result = sentiment_chain.invoke({
    "text": "응답 속도는 빠르지만 설명이 조금 부족했어요."
})

sentiment_result


SentimentResult(sentiment='neutral', reason='응답 속도에 대해서는 긍정적으로 평가하는 반면, 설명이 부족한 점을 지적하여 긍정과 부정이 혼합된 표현이다.', confidence=0.85)

In [14]:
sentiment_result.model_dump()


{'sentiment': 'neutral',
 'reason': '응답 속도에 대해서는 긍정적으로 평가하는 반면, 설명이 부족한 점을 지적하여 긍정과 부정이 혼합된 표현이다.',
 'confidence': 0.85}

# 6. 대표적인 프롬프트 기법

## (1) Zero-shot 프롬프팅

- Zero-shot은 예시 없이 작업 지시만으로 결과를 요청하는 방식
- 작업이 단순하거나 분류 기준이 명확할 때 유용


In [13]:
zero_shot_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        '입력 문장을 다음 중 하나로 분류하세요.: 버그, 기능, 요청, 결제, 사용법, 기타.'
        '분류명만 출력하세요.'
    ),
    ("user", "{message}"),
])

zero_shot_chain = zero_shot_prompt | model | parser

print(zero_shot_chain.invoke({
    "message": "대시보드에서 엑셀 다운로드 버튼을 눌러도 아무 반응이 없습니다."
}))


버그


## (2) Few-shot 프롬프팅

- Few-shot은 원하는 입력/출력 예시를 함께 제공하는 방식
- 모델이 답변 패턴을 따라야 할 때 유용


In [15]:
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "입력 문장이 AI와 관련 있으면 true, 관련 없으면 false만 출력하세요."),
    ("user", "AI 모델의 추론 비용이 증가했다"),
    ("assistant", "true"),
    ("user", "오늘 점심은 김치찌개를 먹었다"),
    ("assistant", "false"),
    ("user", "벡터 데이터베이스로 문서를 검색한다"),
    ("assistant", "true"),
    ("user", "{sentence}"),
])

few_shot_chain = few_shot_prompt | model | parser

print(few_shot_chain.invoke({"sentence": "주말에 등산을 다녀왔다"}))


false


## (3) 근거 요청과 검산 요청

- 복잡한 계산이나 판단에서는 답만 요구하면 오류를 발견하기 어려움
- 다만 내부 사고 과정을 길게 요구하기보다는, 사용자가 검토할 수 있는 **짧은 근거, 계산식, 검산 결과**를 요청하는 것이 실무적으로 더 좋음


In [ ]:
answer_only_prompt = ChatPromptTemplate.from_messages([
    ('system','정답만 출력하세요'),
    ('user','10 + 2 * 3 - 4 * 2의 값을 계산하세요.')
])

answer_only_chain = answer_only_prompt | model | parser
print(answer_only_chain.invoke({}))


10


In [24]:
answer_only_prompt = ChatPromptTemplate.from_messages([
    ('system','1차로 계산하고 계산 과정을 보여줘 그리고 2차로 검산까지해. 마지막으로 정답을 출력하세요'),
    ('user','10 + 2 * 3 - 4 * 2의 값을 계산하세요.')
])

answer_only_chain = answer_only_prompt | model | parser
print(answer_only_chain.invoke({}))


### 1차 계산 과정
주어진 식은:  
10 + 2 * 3 - 4 * 2

곱셈을 먼저 계산합니다:  
2 * 3 = 6  
4 * 2 = 8

식은 이제:  
10 + 6 - 8

덧셈과 뺄셈을 왼쪽에서 오른쪽으로 계산합니다:  
10 + 6 = 16  
16 - 8 = 8

### 2차 검산
원래 식: 10 + 2 * 3 - 4 * 2  
곱셈 부분:  
2 * 3 = 6  
4 * 2 = 8

따라서, 10 + 6 - 8  
먼저 10 + 6 = 16  
그 다음 16 - 8 = 8

### 정답
8


## (4) 작업 분해

- 한 번에 큰 작업을 요청하면 결과가 흔들릴 수 있음
- 작업을 작은 단계로 나누면 더 안정적인 결과를 얻을 수 있음


In [26]:
decompose_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 업무 자동화 컨설턴트다."),
    (
        "user",
        "작업: {task}\n\n"
        "아래 형식으로 작업을 분해해줘.\n"
        "1. 입력 데이터\n"
        "2. 처리 단계\n"
        "3. 출력물\n"
        "4. 실패 가능성과 대응 방법",
    ),
])

decompose_chain = decompose_prompt | model | parser

print(decompose_chain.invoke({
    "task": "여행사 문의가 들어왔는데 65세 조금 전에 은퇴하신 어르신 남성. 강성으로 대응할 때가 많고 이번에는 줄기차게 우리 여행사를 이용해주시다가 키르기스스탄 트레킹 여행 패키지를 우리 회사 통해서 간다고 하신다. 여름이 메인 타겟인 이 나라 여행을 어떻게 하면 좋을까?"
}))


1. 입력 데이터
- 고객 정보: 65세 전후 은퇴한 남성, 강성 성격
- 고객 관계: 꾸준한 단골, 신뢰 기반
- 여행 목적지: 키르기스스탄
- 여행 테마: 트레킹 패키지
- 시즌: 여름(메인 타겟 시즌)

2. 처리 단계
1) 고객 이해 및 심리 파악:
  - 고객의 강성 성격과 긴밀한 관계를 고려한 맞춤형 소통 전략 수립
  - 은퇴 시기와 관심사(트레킹 등)에 맞춘 여행 컨셉 준비
2) 키르기스스탄 여름 트레킹 정보 수집:
  - 여름철 날씨, 트레킹 코스, 인기 명소, 안전 정보 등 확보
3) 패키지 커스터마이징:
  - 고객의 체력과 성향에 맞는 일정 조정
  - 여행 일정 중 편안한 휴식 시간 및 여유 공간 포함
  - 특별 서비스(가이드 배정, 현지 체험 등) 고려
4) 커뮤니케이션 전략 수립:
  - 신뢰를 유지하는 동시에 고객의 요구에 탄력적으로 대응할 방법 설계
  - 정기적인 정보 제공 및 문의 대응 플랜 작성
5) 내부 학습 및 직원 교육:
  - 강성 고객 대응 방안 교육
  - 키르기스스탄 여행 전문성 강화

3. 출력물
- 맞춤형 키르기스스탄 여름 트레킹 여행 패키지 제안서
- 고객별 대응 매뉴얼(성격별 커뮤니케이션 팁 포함)
- 여행 일정표 및 고객 맞춤 서비스 리스트
- 여행사 내부 교육 자료 및 대응 프로토콜

4. 실패 가능성과 대응 방법
- 실패 가능성 1: 고객 불만 및 신뢰 저하
  대응 방법: 고객 의견 적극 반영, 문제 발생 시 신속하고 투명한 소통
- 실패 가능성 2: 트레킹 일정이 고객 체력과 맞지 않음
  대응 방법: 사전 체력 평가, 일정 조정 및 대안 제공
- 실패 가능성 3: 현지 안전 문제 발생
  대응 방법: 안전 정보 최신화, 가이드 교육 강화, 비상 대응 계획 수립
- 실패 가능성 4: 패키지 가격 및 조건 불만족
  대응 방법: 가격 투명성 확보, 유연한 조건 조정 및 추가 혜택 제공


# 7. 프롬프트 품질 개선 패턴

- 좋은 프롬프트는 한 번에 완성되기보다 다음 과정을 반복하며 개선됨

    1. 원하는 결과 기준을 정한다.
    2. 첫 프롬프트를 작성한다.
    3. 여러 테스트 입력으로 실행한다.
    4. 실패 유형을 기록한다.
    5. 역할, 제약, 출력 형식, 예시를 보강한다.



In [27]:
# 아래는 나쁜 프롬프트를 점검하고 개선하는 예시입니다.
bad_prompt = "하하하하하하 나는 웃기지 않다. 그런데 웃기고 싶다."

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 프롬프트 품질을 점검하는 프롬프트 엔지니어다."),
    (
        "user",
        "다음 프롬프트의 문제점을 지적하고 개선안을 작성해줘.\n\n"
        "프롬프트:\n{prompt}\n\n"
        "출력 형식:\n"
        "[문제점]\n- ...\n\n[개선 프롬프트]\n...\n\n[테스트할 입력]\n- ...",
    ),
])

review_chain = review_prompt | model | parser

print(review_chain.invoke({"prompt": bad_prompt}))


[문제점]
- 프롬프트 문장이 모호하고 불분명하여 요구사항이 명확하지 않음.
- "나는 웃기지 않다. 그런데 웃기고 싶다." 라는 문장에서 사용자 의도(예: 웃긴 문장 생성, 코칭, 조언 등)가 드러나지 않아 모델이 어떻게 응답해야 할지 판단하기 어려움.
- 프롬프트가 너무 짧고 구체적인 지시가 없기 때문에 출력 결과가 다양하게 나올 수 있음.
- 유머 스타일, 목적, 형식 등에 대한 추가 정보가 없어서 원하는 방향성을 제공하지 못함.

[개선 프롬프트]
"나는 스스로 웃기지 않다고 생각하지만, 웃기고 싶습니다. 웃긴 농담이나 재치 있는 말을 몇 개 만들어 주세요. 유머 스타일은 가볍고 긍정적인 방향으로 부탁합니다."

[테스트할 입력]
- "나는 평소에 농담을 잘 못해서 사람들 앞에서 웃긴 말을 하고 싶어."
- "유머를 배우고 싶은데 어떻게 해야 할지 모르겠어."
- "친구들과 웃기게 대화할 수 있는 간단한 농담을 알려줘."
